# 调优自动特征工程
在`自动特征工程`中,我们的df字段类型都是由woodwork自动推断的，几乎是完全自动化的过程。 

我们需要对字段类型引入更多的人为设置
- 时间序列
- 自定义原语
- 类型纠正

## 导入

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import featuretools as ft
import woodwork as ww
from woodwork.column_schema import ColumnSchema
from woodwork.logical_types import NaturalLanguage, Datetime,Boolean
from featuretools.primitives import AggregationPrimitive, TransformPrimitive
from featuretools.tests.testing_utils import make_ecommerce_entityset
import warnings
warnings.filterwarnings('ignore')
import gc
gc.enable()

print(f'ft: {ft.__version__},  ww: {ww.__version__}')

In [ ]:
application_train = pd.read_csv('data/application_train.csv')
application_test = pd.read_csv('data/application_test.csv')
bureau = pd.read_csv('data/bureau.csv')
bureau_balance = pd.read_csv('data/bureau_balance.csv')
credit_card_balance = pd.read_csv('data/credit_card_balance.csv')
installments_payments = pd.read_csv('data/installments_payments.csv')
previous_application = pd.read_csv('data/previous_application.csv')
pos_cash_balance = pd.read_csv('data/POS_CASH_balance.csv')


In [ ]:
application_train.dtypes.unique()

In [ ]:
ww.logical_types

## woodwork认识：
- 物理类型
- 逻辑类型：
- 语义标签：额外数据含义

逻辑类型是必须的， 语义标签是可选的

woodwork使用了Pandas 的 Accessor机制，这是扩展接口，在`import featuretools`时候，就把ww加上去了

woodwork初始时候会为其添加逻辑类型，语义标签

### 语义标签

In [ ]:
ww.list_semantic_tags()

- `numeric`, `category` 标准语义标签和特定的逻辑类型关联
- `index`,`time_index` woodwork为一些索引列添加标签，表明一些含义
- `date_of_birth` 表明应该解释为出生日期
- `ignore`,`passthrough` 应该被忽略，在ft过程中

我们应该添加额外标签帮助解释

### 逻辑类型

In [ ]:
ww.list_logical_types()

#### unknown类型
当woodwork类型推导没能成功，就设置`unknown`.  我们可以手动设置他

比如， 下面例子，国家代码没有推导类型成功，就设置了Unknown. 我们可以手动设置`CountryCode`

In [ ]:
s = pd.Series(['AU', 'US', 'UA'])
unkown_series = ww.init_series(s)
unkown_series.ww

In [ ]:
countrycode_series = ww.init_series(unkown_series, 'CountryCode')
countrycode_series.ww

key

#### IntegerNullable 
表明是整数，但是可能会有空值，要小心

In [ ]:
series = pd.Series([1, 2, None, 4], dtype="Int64")
intn_series = ww.init_series(series)
intn_series.ww

#### ordinal 有序类型
评分，排名等

## 处理

我们根据自动推导的，在进行微调

### application

In [ ]:
application_train['set'] = 'train'
application_test['set'] = 'test'
application_test['TARGET'] = np.nan
print(application_train.shape, application_test.shape)
app = pd.concat([application_train, application_test], ignore_index=True)

In [ ]:
app_target = app[['SK_ID_CURR', 'TARGET']]
app_set = app[['SK_ID_CURR', 'set']]

In [ ]:
app.ww.init(name= 'app', index='SK_ID_CURR')

In [ ]:
app.ww.name

In [ ]:
app.ww.schema

In [ ]:
app_types = {} # 记录正确逻辑类型

1. TARGET,set 不参与特征生成, 因此设置ignore

In [ ]:
app.ww.set_types(
    semantic_tags={
        'TARGET': 'ignore',
        'set': 'ignore'
    }
)

2. flag, is_not 设置为bool

In [ ]:
FLAG_DOCUMENTS = { f'FLAG_DOCUMENT_{i}':'Boolean' for i in range(2, 22)}
app.ww.set_types(
    logical_types = {
        'FLAG_MOBIL': 'Boolean',
        'FLAG_EMP_PHONE': 'Boolean',
        'FLAG_WORK_PHONE': 'Boolean',
        'FLAG_CONT_MOBILE': 'Boolean',
        'FLAG_EMAIL': 'Boolean',
        'FLAG_PHONE': 'Boolean',
        'REG_CITY_NOT_LIVE_CITY': 'Boolean',
        'REG_CITY_NOT_WORK_CITY': 'Boolean',
        'LIVE_CITY_NOT_WORK_CITY': 'Boolean',
        **FLAG_DOCUMENTS
    }
)

3. 评级

对于一些异常的，我们可以代替为np.nan， ft是不会处理的

In [ ]:
app['REGION_RATING_CLIENT'].unique()

In [ ]:
app['REGION_RATING_CLIENT_W_CITY'].unique()

In [ ]:
app['REGION_RATING_CLIENT_W_CITY'][app['REGION_RATING_CLIENT_W_CITY'] == -1]

In [ ]:
app['REGION_RATING_CLIENT_W_CITY'] = app['REGION_RATING_CLIENT_W_CITY'].replace(-1, np.nan)

In [ ]:
app.ww.set_types(
    logical_types = {
        'REGION_RATING_CLIENT': ww.logical_types.Ordinal(order=[1,2,3]),
        'REGION_RATING_CLIENT_W_CITY': ww.logical_types.Ordinal(order=[1,2,3]),
    }
)

4. 时间段，应该为分类

In [ ]:
app.ww.set_types(
    logical_types = {
        'HOUR_APPR_PROCESS_START': 'Categorical',
    }
)

### bureau

In [ ]:
bureau.ww.init(name= 'bureau', index='SK_ID_BUREAU')

In [ ]:
bureau.ww.schema

1. id不参与

In [ ]:
bureau.ww.set_types(semantic_tags={'SK_ID_CURR':'ignore'})

In [ ]:
bureau_balance = bureau_balance.reset_index().rename(columns = {'index':'bureaubalance_index'})
bureau_balance.ww.init(name='bureau_balance', index='bureaubalance_index')

In [ ]:
bureau_balance.ww.schema

In [ ]:
bureau_balance.ww.set_types(semantic_tags={'SK_ID_BUREAU':'ignore'})

### previous

In [ ]:
previous_application.ww.init(name='previous', index='SK_ID_PREV')

In [ ]:
previous_application.ww.schema

In [ ]:
previous_application.ww.set_types(semantic_tags={'SK_ID_CURR':'ignore'})

In [ ]:
previous_application.ww.set_types(
    logical_types = {
        'HOUR_APPR_PROCESS_START': 'Categorical',
    }
)

In [ ]:
previous_application['NFLAG_LAST_APPL_IN_DAY'].unique()

In [ ]:
previous_application['NFLAG_INSURED_ON_APPROVAL'].unique()

In [ ]:
previous_application['NFLAG_INSURED_ON_APPROVAL'].isnull().sum()

nan比例过大，我们不应该设置`NFLAG_INSURED_ON_APPROVAL` 为布尔, 而是Categorical

In [ ]:
previous_application.ww.set_types(
    logical_types = {
        'NFLAG_LAST_APPL_IN_DAY': 'Boolean',
        'NFLAG_INSURED_ON_APPROVAL': 'Categorical'
        }
)

In [ ]:
credit_card_balance = credit_card_balance.reset_index().rename(columns = {'index':'credit_index'})
credit_card_balance.ww.init(name='credit', index = 'credit_index')

In [ ]:
credit_card_balance.ww.schema

In [ ]:
credit_card_balance.ww.set_types(semantic_tags={'SK_ID_CURR':'ignore', 'SK_ID_PREV':'ignore'})

In [ ]:
installments_payments = installments_payments.reset_index().rename(columns = {'index':'installments_index'})

installments_payments.ww.init(name = 'installments', index='installments_index')

In [ ]:
installments_payments.ww.schema

In [ ]:
installments_payments.ww.set_types(semantic_tags={'SK_ID_CURR':'ignore', 'SK_ID_PREV':'ignore'})

`NUM_INSTALMENT_VERSION` 更换类型为整数，这也会影响到后面得特征矩阵

In [ ]:
installments_payments.ww.set_types(
    logical_types = {
        'NUM_INSTALMENT_VERSION': 'Integer',
        'DAYS_INSTALMENT': 'Integer'
    }
)

In [ ]:
installments_payments['DAYS_INSTALMENT'].isnull().sum()

In [ ]:
installments_payments['NUM_INSTALMENT_VERSION'].isnull().sum()

In [ ]:
pos_cash_balance = pos_cash_balance.reset_index().rename(columns = {'index':'cash_index'})
pos_cash_balance.ww.init(name='cash', index='cash_index')


In [ ]:
pos_cash_balance.ww.schema

In [ ]:
pos_cash_balance.ww.set_types(semantic_tags={'SK_ID_CURR':'ignore', 'SK_ID_PREV':'ignore'})

## 关系

featuretools对于已经初始化的ww，有些要求：
- 具备index
- name

In [ ]:
es = ft.EntitySet(id='clients')

# 有主键唯一列
es = es.add_dataframe( dataframe=app,)
es = es.add_dataframe( dataframe=bureau,)
es = es.add_dataframe( dataframe=previous_application)

# 没有主键唯一的列，需要make_index, 创建一列主键
es = es.add_dataframe(dataframe=bureau_balance)
es = es.add_dataframe( dataframe=credit_card_balance)
es = es.add_dataframe( dataframe=installments_payments)
es = es.add_dataframe(dataframe=pos_cash_balance)

In [ ]:
# 父亲dfname, 父亲列名； 字dfname, 子列名
es = es.add_relationship("app", "SK_ID_CURR", "bureau", "SK_ID_CURR")
es = es.add_relationship("bureau", "SK_ID_BUREAU", "bureau_balance", "SK_ID_BUREAU")

es = es.add_relationship("app", "SK_ID_CURR", "previous", "SK_ID_CURR")
es = es.add_relationship("previous", "SK_ID_PREV", "cash", "SK_ID_PREV")
es = es.add_relationship("previous", "SK_ID_PREV", "installments", "SK_ID_PREV")
es = es.add_relationship("previous", "SK_ID_PREV", "credit", "SK_ID_PREV")

在构建完关系后，语义标签上会携带外键

In [ ]:
es['app'].ww

In [ ]:
es['bureau'].ww

In [ ]:
es['bureau_balance'].ww

In [ ]:
es['previous'].ww

In [ ]:
es['cash'].ww

In [ ]:
es['credit'].ww

In [ ]:
es['installments'].ww

## 添加interesting values
就是where = 条件聚合。

比如设置agg原语`mean`,  产生`MEAN(prev.AMT_CREDIT)`

如果另外设置where原语`count` 和 兴趣 `{"NAME_CONTRACT_STATUS": ["Approved", "Refused"]}` 
就会多两个特征
- `COUNT(prev.AMT_CREDIT where NAME_CONTRACT_STATUS==Approved) `
- `COUNT(prev.AMT_CREDIT where NAME_CONTRACT_STATUS==Refused) `


In [ ]:
es.add_interesting_values(dataframe_name='previous', values= {
    "NAME_CONTRACT_STATUS": ["Approved", "Refused"]
})

In [ ]:
es['previous'].ww.columns['NAME_CONTRACT_STATUS'].metadata

我们确实为这个列添加了where值

## seed feature
where 条件聚合


In [ ]:
# 在homecredit的逾期还款情况
FLAG_LATED = ft.Feature(es['installments'].ww['DAYS_ENTRY_PAYMENT']) > ft.Feature(es['installments'].ww['DAYS_INSTALMENT'])

FLAG_LATED标识了一种条件

放入` ft.dfs(seed_features=[FLAG_LATED], where_primitives=['count', 'mean'])`, dfs会产生
- `COUNT(installments WHERE FLAG_LATED = True)`
- `MEAN(installments WHERE FLAG_LATED = True)`

In [ ]:
# 在其他机构的逾期还款情况
FLAG_DUE = ft.Feature(es['bureau_balance'].ww['STATUS']).isin(['1', '2', '3', '4', '5'])

In [ ]:
FLAG_DUE

## 自定义特征原语

In [ ]:
es['previous'].ww['NAME_CONTRACT_STATUS'].value_counts().sum()

In [ ]:
class NormalizedModeCount(AggregationPrimitive):
    """ 计算出现最多的次数占比总数的比例。
    """
    name = 'normalized_mode_count'
    input_types = [ColumnSchema(semantic_tags={'category'})]
    return_type = ColumnSchema(semantic_tags={"numeric"})

    def get_function(self):
        def normalized_mode_count(column):
            if len(column) == 0:
                return 0
            counts = column.value_counts()
            if len(counts) == 0:
                return 0
            return counts.max()/counts.sum()
        return normalized_mode_count

比如对于NAME_CONTRACT_STATUS， 表明以往 申请 通过或者拒绝的比例

In [ ]:
class MaxConsecutive(AggregationPrimitive):
    """ 最大连续次数，一般针对bool
    """
    name = 'max_consecutive'
    input_types = [ColumnSchema(logical_type=Boolean)]
    return_type = ColumnSchema(semantic_tags={"numeric"})
    def get_function(self):
        def max_consecutive(column):
            if len(column) == 0:
                return 0
            s = "".join(column.astype(int).astype(str))
            
            segments = s.split("0")
            return max([len(seg) for seg in segments]) if segments else 0
        return max_consecutive

## dfs

ignore_variables
除了'ignore', 这里我们也设置，确保原语不会操作这些特征。

外键和索引不需要管

In [ ]:
%%time
default_agg_primitives = ["count", "mean", "max", "sum", "std"]
default_trans_primitives =  ["month", "weekday"]

# 返回特征矩阵； 特征
feature_matrix, features = ft.dfs(
    entityset = es,
    target_dataframe_name = 'app', # 最后要关联到这个表，以这个为主
    agg_primitives= default_agg_primitives + [NormalizedModeCount, MaxConsecutive],
    trans_primitives=default_trans_primitives,
    max_depth=2,
    seed_features=[FLAG_LATED, FLAG_DUE],
    where_primitives=['mean'],
    ignore_columns  = {
        'app':['set']
    }
)

In [ ]:
[f for f in features if 'set' in f.get_name().lower()]

In [ ]:
feature_matrix.to_parquet("ft_tuning_feature_matrix.parquet")
ft.save_features(features, "ft_tuning_feature_definitions.json")

耗时1h30min

我们需要检查我们的特征确实生效了

In [ ]:
[f for f in features if f.primitive.name == 'normalized_mode_count']

In [ ]:
[f for f in features if f.primitive.name == 'max_consecutive']

In [ ]:
[col for col in feature_matrix.columns if 'WHERE' in col]

？？ 我们没有看到种子特征，`FLAG_LATED`和`FLAG_DUE`

## modeling
- lgbm不需要one-hot

In [ ]:
feature_matrix = pd.read_parquet("ft_tuning_feature_matrix.parquet")

In [ ]:
final_fm = feature_matrix.reset_index()

In [ ]:
final_fm['TARGET']

In [ ]:
final_fm = pd.merge(final_fm, app_set, on='SK_ID_CURR', how='left')

train = final_fm[final_fm['set'] == 'train']
test = final_fm[final_fm['set'] == 'test']

train, test = train.align(test, join = 'inner', axis = 1)
train = train.drop(columns=['set'])
test = test.drop(columns = ['TARGET', 'set'])
print(train.shape, test.shape)

In [ ]:
train_labels = train['TARGET']
train_ids = train['SK_ID_CURR']
test_ids = test['SK_ID_CURR']
train_features = train.drop(columns=['TARGET', 'SK_ID_CURR'])
test_features = test.drop(columns=['SK_ID_CURR'])

In [ ]:
import re
# 1. 定义清理函数
def clean_names(df):
    # 替换所有非字母、数字的字符为下划线
    # 这里的正则 [^A-Za-z0-9_] 会匹配空格、斜杠、括号等所有特殊字符
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]
    # 顺便处理一下可能出现的重复下划线，比如 __
    df.columns = [re.sub(r'_+', '_', col).strip('_') for col in df.columns]
    return df
    

In [ ]:
train_features = clean_names(train_features)
test_features = clean_names(test_features)

from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(
    n_estimators=100,      # 对应 max_iter，树的个数
    learning_rate=0.1,     # 学习率
    max_depth=3,           # 树的最大深度
    random_state=42,       # 保证结果可复现
    n_jobs=-1              # 使用所有 CPU 核心加速
)
lgbm_model.fit(train_features, train_labels)


In [ ]:
features_importance = pd.DataFrame(
    {
        'importance': lgbm_model.feature_importances_,
        'feature': lgbm_model.feature_name_
    }
)
features_importance_plot = features_importance.sort_values(by='importance', ascending=False).head(20)

plt.figure(figsize=(8, 6), dpi=100) 
sns.barplot(data=features_importance_plot, x='importance', y='feature')

plt.yticks(fontsize=7) # 进一步微调
plt.title('Feature Importance', fontsize=14)
plt.tight_layout()

In [ ]:
import time
import os

def submit(ids, pred, name, feature_count=None):
    """
    ids: 测试集的 SK_ID_CURR
    pred: 模型预测概率
    name: 你的实验备注 (如 'lgb_v1', 'baseline')
    feature_count: 可选，记录模型使用了多少个特征
    """
    # 1. 创建提交 DataFrame
    submit_df = pd.DataFrame({
        'SK_ID_CURR': ids,
        'TARGET': pred
    })

    # 2. 生成时间戳 (格式: 0213_1530)
    timestamp = time.strftime("%m%d_%H%M")
    
    # 3. 构造文件名
    # 格式: 0213_1530_lgb_v1_f542.csv
    f_str = f"_f{feature_count}" if feature_count else ""
    filename = f"{timestamp}_{name}{f_str}.csv"
    
    # 4. 确保保存目录存在 (可选)
    if not os.path.exists('submissions'):
        os.makedirs('submissions')
    
    save_path = os.path.join('submissions', filename)
    
    # 5. 保存并打印提示
    submit_df.to_csv(save_path, index=False)
    
    return submit_df


In [ ]:
lgbm_model_pred = lgbm_model.predict_proba(test_features)

In [ ]:
submit_df = submit(test['SK_ID_CURR'], lgbm_model_pred[:, 1], 
    name='lgbm_baseline',
    feature_count=train_features.shape[1]
    )
submit_df.head()


得分76，差不多